# Measurement instrument for LOC

This notebook demonstrates the basic concepts of the measurement instrument. It uses two type of vocabulary - one from the measurement community and one from the software engineering community. 

The goal of this demo is to illustrate the basic concepts:
* measured entity
* measured attribute
* measurement instrument
* base measure

The further processing of the base measure is similar to the one with defects, i.e. assignment 1. 

## Part 1: Configuration

In [1]:
# measured entity is also the input to the measurement instrument
# however, this just just a name of it
measured_entity_name = './Program.cs'

# for non-measurement vocabulary, we would probably name this variable like this:
input_file_name = './Program.cs'

# base measure
base_measure = 0

In [ ]:
# Import required modules for SQLite database
import sqlite3
import datetime
import os

In [ ]:
# Database setup and table creation
def setup_database():
    """Create SQLite database and measurement results table if they don't exist"""
    conn = sqlite3.connect('measurement_results.db')
    cursor = conn.cursor()
    
    # Create table for storing measurement results
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS loc_measurements (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            file_name TEXT NOT NULL,
            file_path TEXT NOT NULL,
            loc_count INTEGER NOT NULL,
            measurement_date TEXT NOT NULL,
            notes TEXT
        )
    ''')
    
    conn.commit()
    conn.close()
    print("Database and table created successfully!")

# Initialize the database
setup_database()

## Part 2: measurement instrument code

In [2]:
# here we have the actual measured entity
measured_entity = open(measured_entity_name, 'r')

# if we wrote the program without the measurement vocabulary, we would probably use something like this:
input_file = open(input_file_name, 'r')

In [3]:
# read the lines into the file
# which means that we "load the attribute here"
measured_attribute = measured_entity.readlines()

In [4]:
# accumulator for LOC
LOC = 0

# we count the lines
for one_line in measured_attribute:
    LOC += 1

## Part 3: assigning the value to the base measure

In [5]:
base_measure = LOC

## Part 4: Further processing

In [6]:
print(f'LOC: {base_measure}')

LOC: 271


## Part 5: Database

In [ ]:
# Function to store measurement results in database
def store_measurement_result(file_path, loc_count, notes=""):
    """Store the measurement result in the SQLite database"""
    conn = sqlite3.connect('./measurement_results.db')
    cursor = conn.cursor()
    
    # Get just the filename from the full path
    file_name = os.path.basename(file_path)
    
    # Get current timestamp
    measurement_date = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Insert the measurement result
    cursor.execute('''
        INSERT INTO loc_measurements (file_name, file_path, loc_count, measurement_date, notes)
        VALUES (?, ?, ?, ?, ?)
    ''', (file_name, file_path, loc_count, measurement_date, notes))
    
    conn.commit()
    conn.close()
    
    print(f"Stored measurement result: {file_name} has {loc_count} LOC")

# Store the current measurement result
store_measurement_result(measured_entity_name, base_measure, "Basic LOC count measurement")

In [ ]:
# Functions to query and display stored results
def display_all_measurements():
    """Display all stored measurement results"""
    conn = sqlite3.connect('measurement_results.db')
    cursor = conn.cursor()
    
    cursor.execute('''
        SELECT id, file_name, file_path, loc_count, measurement_date, notes
        FROM loc_measurements
        ORDER BY measurement_date DESC
    ''')
    
    results = cursor.fetchall()
    conn.close()
    
    if results:
        print("=== All Measurement Results ===")
        print(f"{'ID':<3} {'File Name':<20} {'LOC':<6} {'Date':<19} {'Notes'}")
        print("-" * 70)
        for row in results:
            id_val, file_name, file_path, loc_count, date, notes = row
            print(f"{id_val:<3} {file_name:<20} {loc_count:<6} {date:<19} {notes}")
    else:
        print("No measurement results found in the database.")

def get_measurement_statistics():
    """Get basic statistics about stored measurements"""
    conn = sqlite3.connect('measurement_results.db')
    cursor = conn.cursor()
    
    cursor.execute('''
        SELECT 
            COUNT(*) as total_measurements,
            AVG(loc_count) as avg_loc,
            MIN(loc_count) as min_loc,
            MAX(loc_count) as max_loc
        FROM loc_measurements
    ''')
    
    stats = cursor.fetchone()
    conn.close()
    
    if stats and stats[0] > 0:
        print("=== Measurement Statistics ===")
        print(f"Total measurements: {stats[0]}")
        print(f"Average LOC: {stats[1]:.2f}")
        print(f"Minimum LOC: {stats[2]}")
        print(f"Maximum LOC: {stats[3]}")
    else:
        print("No statistics available - no measurements in database.")

# Display current results
display_all_measurements()
print()
get_measurement_statistics()

In [ ]:
# Example: Measure multiple files and store results
def measure_multiple_files(file_paths):
    """Measure LOC for multiple files and store results"""
    for file_path in file_paths:
        try:
            # Check if file exists
            if not os.path.exists(file_path):
                print(f"Warning: File {file_path} not found, skipping...")
                continue
                
            # Measure the file
            with open(file_path, 'r') as file:
                lines = file.readlines()
                loc_count = len(lines)
            
            # Store in database
            store_measurement_result(file_path, loc_count, f"Batch measurement")
            
        except Exception as e:
            print(f"Error measuring {file_path}: {str(e)}")

# Example usage - measure multiple files if they exist
example_files = ['./Program.cs', './main.c']
available_files = [f for f in example_files if os.path.exists(f)]

if available_files:
    print("=== Measuring multiple files ===")
    measure_multiple_files(available_files)
    print()
    # Display updated results
    display_all_measurements()
else:
    print("No additional files found for batch measurement example.")